# Imports

The workload functions live in `threads_vs_processes/workloads.py`. A process started with
"spawn" (Windows, macOS) cannot use a function defined inside a notebook, so an earlier version of the process cell failed there.
`PCB_NOTEBOOK_N` lets CI run a smaller size; the full size is 1e7.

In [ ]:
import multiprocessing
import os
import threading
import time

from threads_vs_processes.workloads import generate_and_add_numbers, generate_and_join_letters

N = int(float(os.environ.get("PCB_NOTEBOOK_N", 1e7)))
SEED = 2025

# Sequential Execution

In [ ]:
print("Starting the Program")
total_start_time = time.time()

generate_and_add_numbers(N, SEED)
generate_and_join_letters(N, SEED + 1)

total_end_time = time.time()
print("Exiting the Program")
sequential_execution_time = total_end_time - total_start_time
print(f"It took {sequential_execution_time}s to execute the tasks.")

# Threads execution (one thread per function)

In [ ]:
print("Starting the  thread program")
total_start_time = time.time()

thread_numbers = threading.Thread(target=generate_and_add_numbers, args=[N, SEED])
thread_letters = threading.Thread(target=generate_and_join_letters, args=[N, SEED + 1])

thread_numbers.start()
thread_letters.start()

thread_numbers.join()
thread_letters.join()

total_end_time = time.time()
print("Exiting the thread program")
thread_execution_time = total_end_time - total_start_time
print(f"It took {thread_execution_time}s to execute the tasks with thread.")

# Processes execution (one process per function)

Earlier bug: this cell started two processes that both ran `generate_and_add_numbers`, so it did not do the
same work as the sequential cell. Now one process runs each function, like the thread cell.
The earlier cells that ran the numbers function in two threads twice (under two labels) were removed.

In [ ]:
if __name__ == "__main__":
    print("Starting the process program")
    total_start_time = time.time()

    process_numbers = multiprocessing.Process(target=generate_and_add_numbers, args=[N, SEED])
    process_letters = multiprocessing.Process(target=generate_and_join_letters, args=[N, SEED + 1])

    process_numbers.start()
    process_letters.start()

    process_numbers.join()
    process_letters.join()

    total_end_time = time.time()
    print("Exiting the process program")
    process_execution_time = total_end_time - total_start_time
    print(f"It took {process_execution_time}s to execute the tasks with processes.")

# Interpretations and conclusions
- Computing the speedups

In [ ]:
speedup_thread = sequential_execution_time/thread_execution_time
speedup_processes = sequential_execution_time/process_execution_time

print(f"The speedup using threads is {speedup_thread}")
print(f"The speedup using processes is {speedup_processes}")

- Computing the Efficiency

Earlier bug: efficiency divided by `np = 4`, but only 2 workers ran. p is the number of workers actually used.

In [ ]:
p = 2  # one thread or process per function
efficiency_thread = speedup_thread/p
efficiency_processes = speedup_processes/p

print(f"The efficiency using threads is {efficiency_thread}")
print(f"The efficiency using processes is {efficiency_processes}")

- Amdahl and Gustafson Laws (P = parallel fraction, assumed 0.99)

Earlier version: Amdahl was computed as 1/(1-P), the limit for infinitely many processors. With p workers it is 1/((1-P) + P/p).

In [ ]:
P = 0.99
amdahl_s = 1/((1-P) + P/p)
print(amdahl_s)

In [ ]:
alpha = 1-P
gustafson = p - alpha*(p-1)
print(gustafson)

## Conclusions

The measured numbers are in `results/01_threads_vs_processes/` and the table in this folder's README (script version: one warm-up, then the median of 5 runs).

- Threads did not speed up these loops at all: both functions are pure Python, and the GIL lets only one thread run Python bytecode at a time.
- One process per function roughly halved the time, and two per function (4 processes on 4 cores) cut it to about a quarter, because each process has its own interpreter and GIL.
- Efficiency must use the number of workers that actually ran (p = 2 here), not the number of cores.
- Amdahl's law with P = 0.99 and p = 2 predicts almost 2x, which matches the processes; it says nothing about threads, because the GIL makes the "parallel" part serial.